# Data Inventory

This notebook inventories the Stata microdata files in the project and determines which files are relevant to the final household-level analytic dataset.

In [1]:
import os
import pandas as pd
from pathlib import Path

data_dir = Path('stata')
files = sorted(data_dir.glob('*.dta'))
print(f'Read {len(files)} Stata files from {data_dir}')
for f in files:
    df = pd.read_stata(f)
    print(f'\nFILE: {f.name}')
    print('rows=', df.shape[0], 'cols=', df.shape[1])
    print('first_columns=', df.columns[:12].tolist())
    print('has_idquest=', 'idquest' in df.columns)

Read 16 Stata files from stata

FILE: S0_General Information.dta
rows= 23709 cols= 25
first_columns= ['idquest', 'Segment_ID', 's0q0', 's0q1', 's0q2', 's0q3', 's0q4', 's0q5', 's0q6', 's0q7', 's0q8', 's0q9']
has_idquest= True

FILE: S10_1_Cattle milk production.dta
rows= 3237 cols= 16
first_columns= ['idquest', 's0q1', 's0q2', 's0q3', 's0q4', 's10q1_1', 's10q1_2', 's10q1_3_1', 's10q1_3_2', 's10q1_3_3', 's10q1_4_1', 's10q1_4_2']
has_idquest= True

FILE: S10_2_Cattle milk production use.dta
rows= 3989 cols= 15
first_columns= ['idquest', 's0q2', 's0q3', 's0q4', 's10q2_1', 's10q2_2', 's10q2_3', 's10q2_4', 's10q2_5', 's10q2_6', 's10q2_7', 's10q2_8']
has_idquest= True

FILE: S10_3_Honey production.dta
rows= 16067 cols= 12
first_columns= ['idquest', 's0q2', 's0q3', 's0q4', 's10q3_1', 's10q3_2', 's10q3_3', 's10q3_4', 's10q3_5', 's10q3_6', 's10q3_7', 'weight']
has_idquest= True

FILE: S11_Animal inputs and services.dta
rows= 26500 cols= 20
first_columns= ['idquest', 's0q1', 's0q2', 's0q3', 's0q4

## Decision table

The following cell classifies each file as keep, exclude, or optional based on relevance to the final household-level project.

In [3]:
key_files = {
    'S0_General Information.dta': 'keep',
    'S2_Land tenure and crops planted.dta': 'keep',
    'S5_Agricultural Inputs.dta': 'keep',
    'S6_Agricultural Practices.dta': 'keep',
    'S9_Number of animals.dta': 'keep',
    'S3_Extension services and agricultural programmes.dta': 'optional',
    'S1_Household members characteristics.dta': 'optional',
    'S4_Funding during 2016_2017.dta': 'optional',
    'S7_1_Agricultural small tools.dta': 'optional',
    'S7_2_Agricultural durable tools.dta': 'optional',
    'S8_A_Production Use,storage and Expenses.dta': 'optional',
    'S8_B_Production Use,Storage and Expenses.dta': 'optional',
    'S11_Animal inputs and services.dta': 'optional',
    'S10_1_Cattle milk production.dta': 'exclude',
    'S10_2_Cattle milk production use.dta': 'exclude',
    'S10_3_Honey production.dta': 'exclude',
}

decision = pd.DataFrame({
    'file': list(key_files.keys()),
    'decision': list(key_files.values())
})
decision

,file,decision
0,S0_General Information.dta,keep
1,S2_Land tenure and crops planted.dta,keep
2,S5_Agricultural Inputs.dta,keep
3,S6_Agricultural Practices.dta,keep
4,S9_Number of animals.dta,keep
5,S3_Extension services and agricultural program...,optional
6,S1_Household members characteristics.dta,optional
7,S4_Funding during 2016_2017.dta,optional
8,S7_1_Agricultural small tools.dta,optional
9,S7_2_Agricultural durable tools.dta,optional


## Rationale for the decisions

This notebook does more than list filenames. It establishes which survey modules can contribute valid household-level predictors, which modules are useful only for sensitivity analysis, and which modules are outside the scope of the research question. The decision is based on four criteria: 
- Relevance to agricultural input use and household profiles, 
- Compatibility with the household identifier `idquest`, 
- Risk of duplicated records after merging, 
- And risk of target leakage.


### Files we will keep in the core analytic dataset

- **`S0_General Information.dta` (household foundation):** This is the anchor file. It supplies `idquest` and basic household or location characteristics. Every selected module will be linked to this file, and the final analytic table will be one row per household.
- **`S2_Land tenure and crops planted.dta` (land and crop predictors):** Land access, land size, tenure, and crop information describe the household's agricultural scale and production choices. These variables are directly relevant for predicting whether a household uses agricultural inputs and for forming agricultural household profiles.
- **`S5_Agricultural Inputs.dta` (target and input details):** This module contains the input-use responses. We will derive the binary classification target from `s5q1_1` (`Yes` = 1, `No` = 0), then retain the other input characteristics as candidate explanatory variables only after checking for target leakage. In particular, variables that directly reveal the same input-use decision will be excluded from the predictor matrix.
- **`S6_Agricultural Practices.dta` (management predictors):** Cropping and farming-practice variables capture how households manage production. They provide context for input adoption and add behavioral information beyond land size alone.
- **`S9_Number of animals.dta` (livestock profile):** Livestock ownership and counts measure the breadth and intensity of household agricultural activity. These variables are useful both as classification predictors and as part of the clustering profile.


### Files that are optional or reserved for sensitivity analysis

Optional does not mean irrelevant. It means that a file is not required for the primary model and will be added only if its structure, missingness, and household-level coverage are acceptable. This keeps the core model interpretable and reduces unnecessary dimensionality.


- **`S3_Extension services and agricultural programmes.dta`:** Extension access may explain input adoption, so it is a valuable enrichment variable. It has repeated household records and must be aggregated by `idquest` before merging. The current preparation script uses it as an optional enrichment; the baseline model can be rerun without it to measure whether extension information improves performance.
- **`S1_Household members characteristics.dta`:** Household size, composition, age, and education can explain resources and decision-making, but this module may contain multiple rows per household. We will use household-level summaries such as member count or age statistics only if aggregation is defensible.
- **`S4_Funding during 2016_2017.dta`:** Financing can influence input use, but it may have substantial missingness and may describe an intermediate outcome rather than a pre-existing predictor. It will be tested separately and excluded if it creates leakage or sharply reduces the usable sample.
- **`S7_1_Agricultural small tools.dta` and `S7_2_Agricultural durable tools.dta`:** Tools and machinery are plausible measures of agricultural intensity, but they can overlap with input-use or mechanisation decisions. They will be considered for clustering or a sensitivity model after checking their timing and duplication.
- **`S8_A_Production Use,storage and Expenses.dta` and `S8_B_Production Use,Storage and Expenses.dta`:** Production, use, storage, and expenses may be useful for describing household profiles, but they are downstream of production decisions and may be incomplete or duplicated across modules. They are therefore not needed for the primary classifier and will be considered only for a clearly labelled clustering or sensitivity analysis.
- **`S11_Animal inputs and services.dta`:** Animal services and inputs are relevant to livestock-oriented profiles, but the module is not essential to the crop/input-adoption target and may contain repeated records. It will be aggregated and added only if livestock-related analysis requires it.


### Files excluded from the current analysis

- **`S10_1_Cattle milk production.dta`, `S10_2_Cattle milk production use.dta`, and `S10_3_Honey production.dta`:** These are specialised production modules. They apply only to households engaged in particular activities, which would introduce a selective subpopulation and additional missingness. Their outcomes are specific to milk or honey rather than the general agricultural input-adoption question. Excluding them prevents the main table from becoming sparse and avoids mixing specialised downstream production outcomes with broad household predictors. They can be revisited only if the research question changes to dairy or honey production.


### Exact use of the selected data

1. **Inventory and inspect:** Read every Stata file, record its row and column counts, inspect the first columns, and verify whether `idquest` is present. This confirms which modules can be linked and identifies files requiring aggregation.
2. **Choose the household grain:** Use `S0_General Information.dta` as the household spine. The final modeling table must contain one row per `idquest`; row counts and duplicate identifiers will be checked after every merge.
3. **Prepare the target:** From `S5_Agricultural Inputs.dta`, encode `s5q1_1` as the binary target. Households without a valid target are removed from supervised learning, and class balance is reported before training.
4. **Prepare module-specific features:** Select land and crop variables from `S2`, management variables from `S6`, and livestock variables from `S9`. Repeated modules such as `S3` and `S9` are aggregated by `idquest`: numeric measures are summed when they represent quantities, while categorical responses use a documented representative value such as the mode.
5. **Merge and clean:** Left-merge the selected modules onto the household spine using `idquest`. Resolve suffixes and duplicate fields, document missingness, encode categorical variables, and remove identifiers plus any variables that reveal the target. The cleaned table is saved as `prepared_household_data.csv`.
6. **Run classification:** Use the retained household, land, crop, practice, extension, and livestock features to compare Logistic Regression, Random Forest or Gradient Boosting, and an Artificial Neural Network. Use the same split and feature set for each model and report accuracy, precision, recall, F1-score, and ROC-AUC where appropriate.
7. **Run clustering:** Use a separate unsupervised feature matrix built from standardized land, crop, livestock, and input-intensity variables. Do not use the classification target to create clusters. Compare cluster solutions using diagnostics such as within-cluster variation or silhouette score, then describe each cluster using its household and agricultural characteristics.
8. **Check improvement:** Apply one improvement method, such as class weighting, feature selection, or hyperparameter tuning. Compare the improved model with the baseline using the same evaluation protocol and report whether the change genuinely improves performance.


This design keeps the primary analysis focused, reproducible, and household-level while preserving plausible modules for controlled sensitivity analyses. The final keep/optional decision should be confirmed after the inventory cell reports actual row counts, identifier uniqueness, missingness, and target coverage.

In [5]:
# Phase 1 audit: validate the household key, target, and merge grain.
core_files = [
    'S0_General Information.dta',
    'S2_Land tenure and crops planted.dta',
    'S5_Agricultural Inputs.dta',
    'S6_Agricultural Practices.dta',
    'S9_Number of animals.dta',
    'S3_Extension services and agricultural programmes.dta',
    'S1_Household members characteristics.dta',
    'S4_Funding during 2016_2017.dta',
    'S7_1_Agricultural small tools.dta',
    'S7_2_Agricultural durable tools.dta',
    'S8_A_Production Use,storage and Expenses.dta',
    'S8_B_Production Use,Storage and Expenses.dta',
    'S11_Animal inputs and services.dta',
]

audit_rows = []
for file_name in core_files:
    module = pd.read_stata(data_dir / file_name)
    audit_rows.append({
        'file': file_name,
        'rows': len(module),
        'unique_idquest': module['idquest'].nunique(dropna=True),
        'duplicate_rows': module['idquest'].duplicated().sum(),
        'missing_idquest': module['idquest'].isna().sum(),
    })

audit = pd.DataFrame(audit_rows)
audit['records_per_household'] = (audit['rows'] / audit['unique_idquest']).round(2)
audit

,file,rows,unique_idquest,duplicate_rows,missing_idquest,records_per_household
0,S0_General Information.dta,23709,23419,289,290,1.01
1,S2_Land tenure and crops planted.dta,16057,16057,0,0,1.00
2,S5_Agricultural Inputs.dta,16057,16057,0,0,1.00
3,S6_Agricultural Practices.dta,16057,16057,0,0,1.00
4,S9_Number of animals.dta,32611,16057,16554,0,2.03
5,S3_Extension services and agricultural program...,64228,16057,48171,0,4.00
6,S1_Household members characteristics.dta,73665,16057,57608,0,4.59
7,S4_Funding during 2016_2017.dta,16057,16057,0,0,1.00
8,S7_1_Agricultural small tools.dta,362203,16057,346146,0,22.56
9,S7_2_Agricultural durable tools.dta,207049,16057,190992,0,12.89


In [7]:
# Validate the proposed target before encoding it.
inputs_audit = pd.read_stata(data_dir / 'S5_Agricultural Inputs.dta')
target_values = inputs_audit['s5q1_1'].value_counts(dropna=False).rename_axis('s5q1_1').reset_index(name='records')
target_values['share'] = (target_values['records'] / len(inputs_audit)).round(4)
print('Raw target values:')
display(target_values)

base_audit = pd.read_stata(data_dir / 'S0_General Information.dta')
base_valid_ids = base_audit.loc[base_audit['idquest'].notna(), 'idquest']
input_ids = inputs_audit['idquest']
print(f'Valid household IDs in S0: {base_valid_ids.nunique()}')
print(f'Household IDs in S5: {input_ids.nunique()}')
print(f'Input households also present in S0: {input_ids.isin(set(base_valid_ids)).sum()} of {input_ids.nunique()}')

Raw target values:


,s5q1_1,records,share
0,Yes,11080,0.69
1,No,4977,0.31


Valid household IDs in S0: 23419
Household IDs in S5: 16057
Input households also present in S0: 16057 of 16057


## Phase 1 conclusion: final data-audit decisions

The audit supports the following decisions for the next phase:

- **Classification target:** Use `s5q1_1` from `S5_Agricultural Inputs.dta`. It contains only `Yes` and `No`: 11,080 households (69%) answered `Yes`, and 4,977 (31%) answered `No`. Therefore, encode `Yes` as 1 and `No` as 0. No target rows need to be removed because the target has no missing values.
- **Target coverage:** The target module contains 16,057 unique household IDs, and all 16,057 are also present in `S0_General Information.dta`. This gives complete target coverage for the selected modeling population.
- **Household spine:** `S0_General Information.dta` remains the reference file, but it must be filtered to non-missing `idquest` and deduplicated before merging. The audit found 290 missing IDs and 289 duplicate rows in this file. The one-to-one modules identify the clean modeling population of 16,057 households.
- **Core files for the baseline dataset:** Use `S0`, `S2`, `S5`, and `S6` as the household-level foundation, and include `S9` after aggregation by `idquest`. `S2`, `S5`, and `S6` each have exactly one record per target household; `S9` has approximately two records per household and cannot be merged directly without creating duplicate households.
- **Optional files:** Do not add optional modules to the first baseline dataset. Evaluate `S3` first as a controlled sensitivity extension, because it has four records per household and must be aggregated. Consider `S1`, `S4`, `S7`, `S8`, and `S11` only after checking feature meaning, missingness, aggregation, and target leakage.
- **Excluded files:** Continue excluding the specialised milk and honey modules because they represent selective production activities rather than the general input-adoption target.


### Phase 1 acceptance criteria

Phase 1 is complete when the next preparation step demonstrates that:

1. The cleaned household spine has unique, non-missing `idquest` values.
2. The baseline merge preserves one row per household.
3. The target contains only the values 0 and 1 with the documented class counts.
4. Every feature has a documented source module and aggregation rule.
5. The optional extension-services model can be compared with the core baseline without changing the test population.